<a href="https://colab.research.google.com/github/PainXD1381/farmedge/blob/main/notebooks/kaggle-studio-server.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎬 OpenSource Clipping Studio — Backend Server

This notebook runs the **FastAPI backend** for OpenSource Clipping Studio.

**How it works:**
1. Run all cells below to start the server
2. Copy the **Public URL** that appears
3. Open the Studio page on GitHub Pages and paste the URL
4. Start clipping!

**Studio URL:** [naufalrizqullah.github.io/opensource-clipping/studio/](https://naufalrizqullah.github.io/opensource-clipping/studio/)

---

## 1. Setup Project

In [1]:
!git clone https://github.com/NaufalRizqullah/opensource-clipping.git /content/opensource-clipping
%cd /content/opensource-clipping

Cloning into '/content/opensource-clipping'...
remote: Enumerating objects: 1628, done.
remote: Counting objects: 100% (623/623), done.
remote: Compressing objects: 100% (120/120), done.
remote: Total 1628 (delta 542), reused 506 (delta 503), pack-reused 1005 (from 2)
Receiving objects: 100% (1628/1628), 75.02 MiB | 37.13 MiB/s, done.
Resolving deltas: 100% (1078/1078), done.
/content/opensource-clipping


In [2]:
%%capture
# Install dependencies
!pip install -r requirements.txt
!pip install pyngrok nest-asyncio aiofiles

In [3]:
# Setup system dependencies
!apt-get -qq update && apt-get -qq install -y ffmpeg

W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


## 2. Configure API Keys

Make sure you have added these secrets in **Add-ons > Secrets**:
- `GOOGLE_API_KEY` — Google Gemini API key
- `NGROK_AUTHTOKEN` — ngrok auth token (get from https://dashboard.ngrok.com)
- `PEXELS_API_KEY` — (optional) for B-roll footage
- `HF_TOKEN` — (optional) for speaker diarization

In [16]:
import os
from pathlib import Path

# Try Kaggle secrets first, then Colab
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    def get_secret(name, default=""):
        try:
            return secrets.get_secret(name) or default
        except:
            return default
    print("✅ Using Kaggle Secrets")
except ImportError:
    try:
        from google.colab import userdata
        def get_secret(name, default=""):
            try:
                return userdata.get(name) or default
            except:
                return default
        print("✅ Using Colab Secrets")
    except ImportError:
        def get_secret(name, default=""):
            return os.environ.get(name, default)
        print("⚠️ Using environment variables")

# Load secrets
GOOGLE_API_KEY = get_secret("GOOGLE_API_KEY")
NGROK_AUTHTOKEN = ""
PEXELS_API_KEY = get_secret("PEXELS_API_KEY")
HF_TOKEN = get_secret("HF_TOKEN")

# Set environment variables
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
if PEXELS_API_KEY:
    os.environ["PEXELS_API_KEY"] = PEXELS_API_KEY
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN

# Create .env file
env_text = f"""# Auto-generated from notebook secrets
GOOGLE_API_KEY={GOOGLE_API_KEY}
PEXELS_API_KEY={PEXELS_API_KEY}
HF_TOKEN={HF_TOKEN}
"""
Path(".env").write_text(env_text, encoding="utf-8")

# Status
print(f"  GOOGLE_API_KEY: {'✅ Set' if GOOGLE_API_KEY else '❌ Missing'}")
print(f"  NGROK_AUTHTOKEN: {'✅ Set' if NGROK_AUTHTOKEN else '❌ Missing'}")
print(f"  PEXELS_API_KEY: {'✅ Set' if PEXELS_API_KEY else '⚪ Not set (optional)'}")
print(f"  HF_TOKEN: {'✅ Set' if HF_TOKEN else '⚪ Not set (optional)'}")

✅ Using Colab Secrets
  GOOGLE_API_KEY: ✅ Set
  NGROK_AUTHTOKEN: ❌ Missing
  PEXELS_API_KEY: ⚪ Not set (optional)
  HF_TOKEN: ⚪ Not set (optional)


In [5]:
# ──────────────────────────────────────────────────────────────────────
# YouTube Cookies Setup (Optional) - Solves yt-dlp 403 Forbidden errors
# ──────────────────────────────────────────────────────────────────────
import os
import shutil

COOKIES_PATH = os.path.abspath("youtube_cookies.txt")
cookie_candidates = [
    "/kaggle/input/datasets/muhammadnaufal/tiktok-secret-cookies/youtube_cookies.txt",
    "/content/youtube_cookies.txt",
    COOKIES_PATH,
]

source_cookie = next((p for p in cookie_candidates if os.path.isfile(p)), None)

if source_cookie:
    if os.path.realpath(source_cookie) != os.path.realpath(COOKIES_PATH):
        shutil.copy(source_cookie, COOKIES_PATH)
    print(f"✅ YouTube cookies aktif: {COOKIES_PATH}")
else:
    print("ℹ️ youtube_cookies.txt tidak ditemukan. Download akan dicoba tanpa cookies.")


ℹ️ youtube_cookies.txt tidak ditemukan. Download akan dicoba tanpa cookies.


## 3. Start Backend Server + Tunnel

This cell starts the FastAPI server and creates an ngrok tunnel.

**Copy the Public URL** and paste it into the Studio page!

In [12]:
!pip install pyngrok

In [17]:
from google.colab import userdata

token = userdata.get("NGROK_AUTHTOKEN")

print("Token found:", bool(token))
print("Token length:", len(token) if token else 0)

Token found: True
Token length: 49


In [75]:
# Skip ngrok authentication to avoid breaking 'Run All' since we are using Cloudflare Tunnel
print("Skipping ngrok configuration. Using Cloudflare Tunnel instead for reliable connection.")

Skipping ngrok configuration. Using Cloudflare Tunnel instead for reliable connection.


In [76]:
# Place-holder to avoid breaking subsequent runs
print("Skipping ngrok connection. Running Cloudflare Tunnel in the next steps...")

Skipping ngrok connection. Running Cloudflare Tunnel in the next steps...


In [30]:
import subprocess
import sys
import time

# Start the backend server in the background
backend = subprocess.Popen([
    sys.executable, "-m", "uvicorn",
    "web.api.app:app",
    "--host", "0.0.0.0",
    "--port", "8000",
    "--log-level", "warning"
])

# Give the backend time to start
time.sleep(5)

# Start Cloudflare Tunnel
subprocess.run([
    "cloudflared", "tunnel",
    "--url", "http://127.0.0.1:8000"
])

KeyboardInterrupt: 

In [31]:
!cloudflared --version

cloudflared version 2026.9.3 (built 2026-09-24-16:07 UTC)


In [32]:
import subprocess
import sys
import time

# Start the FastAPI backend in the background
backend = subprocess.Popen([
    sys.executable, "-m", "uvicorn",
    "web.api.app:app",
    "--host", "0.0.0.0",
    "--port", "8000",
    "--log-level", "warning"
])

# Wait 5 seconds for the backend to start
time.sleep(5)

# Check whether the backend process is still running
print("Backend process started:", backend.poll() is None)

Backend process started: True


In [33]:
import subprocess

# Start Cloudflare Tunnel in the background
tunnel = subprocess.Popen([
    "cloudflared", "tunnel",
    "--url", "http://127.0.0.1:8000"
])

print("Cloudflare Tunnel process started.")

Cloudflare Tunnel process started.


In [34]:
import subprocess

result = subprocess.run(
    ["pgrep", "-af", "cloudflared"],
    capture_output=True,
    text=True
)

print(result.stdout or "No Cloudflare process found.")

24490 cloudflared tunnel --url http://127.0.0.1:8000



In [77]:
import subprocess
import time
import re

# Ensure any previous cloudflared processes are killed first
subprocess.run(["pkill", "-f", "cloudflared"], check=False)
time.sleep(2)

# Start Cloudflare Tunnel and log its output
log_file_path = "/tmp/cloudflared_studio.log"
with open(log_file_path, "w") as log_file:
    tunnel_process = subprocess.Popen([
        "cloudflared", "tunnel",
        "--url", "http://127.0.0.1:8000"
    ], stdout=log_file, stderr=subprocess.STDOUT, start_new_session=True)

print("Starting Cloudflare Tunnel...")
time.sleep(8)

# Read the logs and find the generated trycloudflare URL
with open(log_file_path, "r") as f:
    log_content = f.read()

match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", log_content)
if match:
    print("\n" + "="*60)
    print("🎉 YOUR PUBLIC STUDIO BACKEND URL:")
    print(match.group(0))
    print("="*60)
    print("\n1. Copy the URL above.")
    print("2. Paste it in the 'API URL' box on the OpenSource Clipping Studio site.")
else:
    print("Tunnel is connecting. Displaying current logs:")
    print(log_content)

Starting Cloudflare Tunnel...

🎉 YOUR PUBLIC STUDIO BACKEND URL:
https://tuesday-scholarship-struct-kelkoo.trycloudflare.com

1. Copy the URL above.
2. Paste it in the 'API URL' box on the OpenSource Clipping Studio site.


In [27]:
!cloudflared tunnel --url http://127.0.0.1:8000

2026-09-28T06:45:52Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-09-28T06:45:52Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-09-28T06:45:56Z INF +--------------------------------------------------------------------------------------------+
2026-09-28T06:45:56Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-09-28T06:45:56Z INF |  https://scotland-arlington-offset-decades.trycloudfla

In [23]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb -O /tmp/cloudflared.deb && dpkg -i /tmp/cloudflared.deb

(Reading database ... 126956 files and directories currently installed.)
Preparing to unpack /tmp/cloudflared.deb ...
Unpacking cloudflared (2026.9.3) over (2026.9.3) ...
Setting up cloudflared (2026.9.3) ...
Processing triggers for man-db (2.12.0-4build2) ...


In [36]:
import os

root = "/content/opensource-clipping"

for folder, dirs, files in os.walk(root):
    dirs[:] = [d for d in dirs if d not in [".git", "__pycache__", ".venv"]]

    for file in files:
        if file.endswith((".py", ".json", ".yaml", ".yml", ".toml", ".env")):
            path = os.path.join(folder, file)
            try:
                with open(path, "r", encoding="utf-8", errors="ignore") as f:
                    for line_no, line in enumerate(f, 1):
                        if "gemini-2.5-flash" in line.lower():
                            print(f"{path}:{line_no}: {line.strip()}")
            except Exception:
                pass

/content/opensource-clipping/outputs/jobs.json:44: "message": "Pipeline gagal: RuntimeError: Gagal memanggil Gemini utama & fallback. Laporan Utama status=503, error=503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}} | Laporan Fallback error=404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use models/gemini-3.8-flash for the latest features and improvements. We recommend you to use the Interactions API (https://ai.google.dev/gemini-api/docs/get-started).', 'status': 'NOT_FOUND'}}",
/content/opensource-clipping/outputs/jobs.json:49: "error": "RuntimeError: Gagal memanggil Gemini utama & fallback. Laporan Utama status=503, error=503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand 

In [38]:
from pathlib import Path
import shutil

config_path = Path("/content/opensource-clipping/clipping/config.py")

# Create a backup before making any changes
backup_path = config_path.with_suffix(".py.backup")
shutil.copy2(config_path, backup_path)

# Read the configuration file
text = config_path.read_text(encoding="utf-8")

old = 'GEMINI_FALLBACK_MODEL = "gemini-2.5-flash"'
new = 'GEMINI_FALLBACK_MODEL = "gemini-3.8-flash"'

if old not in text:
    print("Old setting not found. No changes made.")
else:
    text = text.replace(old, new, 1)
    config_path.write_text(text, encoding="utf-8")
    print("Fallback model updated successfully!")

print("\nCurrent setting:")
for line in config_path.read_text(encoding="utf-8").splitlines():
    if "GEMINI_FALLBACK_MODEL" in line:
        print(line)

Old setting not found. No changes made.

Current setting:
GEMINI_FALLBACK_MODEL = "gemini-3.8-flash"
        default=GEMINI_FALLBACK_MODEL,


In [39]:
import subprocess

subprocess.run(
    ["pkill", "-f", "uvicorn web.api.app:app"],
    check=False
)

CompletedProcess(args=['pkill', '-f', 'uvicorn web.api.app:app'], returncode=0)

In [40]:
import subprocess
import sys
import time

backend = subprocess.Popen([
    sys.executable, "-m", "uvicorn",
    "web.api.app:app",
    "--host", "0.0.0.0",
    "--port", "8000",
    "--log-level", "warning"
])

time.sleep(5)

print("Backend process running:", backend.poll() is None)

Backend process running: True


In [41]:
import numpy as np
import sys

print("Python version:", sys.version)
print("NumPy version:", np.__version__)
print("NumPy location:", np.__file__)

try:
    print("StringDType available:", hasattr(np.dtypes, "StringDType"))
except AttributeError:
    print("NumPy dtypes module is unavailable")

Python version: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
NumPy version: 2.1.3
NumPy location: /usr/local/lib/python3.13/dist-packages/numpy/__init__.py
StringDType available: True


In [42]:
import importlib.metadata as md

packages = ["numpy", "pandas", "pyarrow", "datasets", "transformers"]

for package in packages:
    try:
        print(f"{package}: {md.version(package)}")
    except md.PackageNotFoundError:
        print(f"{package}: Not installed")

numpy: 1.26.4
pandas: 2.2.3
pyarrow: 23.0.1
datasets: 4.8.5
transformers: 5.16.1


In [43]:
import sys
import numpy as np
import importlib.metadata as md

print("Python executable:", sys.executable)
print("NumPy imported version:", np.__version__)
print("NumPy file:", np.__file__)

try:
    dist = md.distribution("numpy")
    print("NumPy metadata version:", dist.version)
    print("NumPy metadata location:", dist.locate_file(""))
except Exception as e:
    print("Metadata error:", e)

print("StringDType available:", hasattr(np.dtypes, "StringDType"))

Python executable: /usr/bin/python3
NumPy imported version: 2.1.3
NumPy file: /usr/local/lib/python3.13/dist-packages/numpy/__init__.py
NumPy metadata version: 1.26.4
NumPy metadata location: /usr/local/lib/python3.13/dist-packages
StringDType available: True


In [44]:
import subprocess
import sys

print("=== pip information ===")
subprocess.run([sys.executable, "-m", "pip", "show", "numpy"])

print("\n=== dependency conflicts ===")
subprocess.run([sys.executable, "-m", "pip", "check"])

=== pip information ===

=== dependency conflicts ===


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'check'], returncode=1)

In [45]:
import subprocess
import sys

for command in [
    [sys.executable, "-m", "pip", "show", "numpy"],
    [sys.executable, "-m", "pip", "check"],
]:
    result = subprocess.run(
        command,
        capture_output=True,
        text=True
    )
    print("\nCOMMAND:", " ".join(command))
    print("RETURN CODE:", result.returncode)
    print("OUTPUT:\n", result.stdout)
    print("ERROR:\n", result.stderr)


COMMAND: /usr/bin/python3 -m pip show numpy
RETURN CODE: 0
OUTPUT:
 Name: numpy
Version: 1.26.4
Summary: Fundamental package for array computing in Python
Home-page: https://numpy.org
Author: Travis E. Oliphant et al.
Author-email: 
License: Copyright (c) 2005-2023, NumPy Developers.
 All rights reserved.

 Redistribution and use in source and binary forms, with or without
 modification, are permitted provided that the following conditions are
 met:

     * Redistributions of source code must retain the above copyright
        notice, this list of conditions and the following disclaimer.

     * Redistributions in binary form must reproduce the above
        copyright notice, this list of conditions and the following
        disclaimer in the documentation and/or other materials provided
        with the distribution.

     * Neither the name of the NumPy Developers nor the names of any
        contributors may be used to endorse or promote products derived
        from this software 

In [47]:
import subprocess
import sys

print("Upgrading numpy...")
subprocess.run([sys.executable, "-m", "pip", "install", "numpy", "--upgrade"])
print("NumPy upgrade complete.")

Upgrading numpy...
NumPy upgrade complete.


In [48]:
import subprocess
import sys

print("\n=== Checking for dependency conflicts again ===")
subprocess.run([sys.executable, "-m", "pip", "check"])


=== Checking for dependency conflicts again ===


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'check'], returncode=1)

In [49]:
import subprocess
import sys

print("=== pip information for numpy after upgrade ===")
result_show = subprocess.run(
    [sys.executable, "-m", "pip", "show", "numpy"],
    capture_output=True,
    text=True
)
print(result_show.stdout)
print("RETURN CODE:", result_show.returncode)
print("ERROR:\n", result_show.stderr)

print("\n=== Full dependency conflicts after upgrade ===")
result_check = subprocess.run(
    [sys.executable, "-m", "pip", "check"],
    capture_output=True,
    text=True
)
print(result_check.stdout)
print("RETURN CODE:", result_check.returncode)
print("ERROR:\n", result_check.stderr)

=== pip information for numpy after upgrade ===
Name: numpy
Version: 2.5.3
Summary: Fundamental package for array computing in Python
Home-page: https://numpy.org
Author: Travis E. Oliphant et al.
Author-email: 
License: 
Location: /usr/local/lib/python3.13/dist-packages
Requires: 
Required-by: accelerate, access, albucore, albumentations, ale-py, arviz, asteroid-filterbanks, astropy, autograd, bigframes, blis, blosc2, bokeh, Bottleneck, bqplot, clarabel, cmdstanpy, contourpy, ctranslate2, cucim-cu12, cuda-core, cudf-cu12, cufflinks, cuml-cu12, cupy-cuda12x, cuvs-cu12, cvxpy, cyipopt, dask-cuda, dask-cudf-cu12, datasets, db-dtypes, diffusers, dm-tree, dopamine_rl, esda, flax, folium, formulaic, geemap, geopandas, gradio, grain, gym, gymnasium, h5netcdf, h5py, hdbscan, highspy, holoviews, hyperopt, ImageIO, imbalanced-learn, inequality, jax, jaxlib, keras, keras-hub, libpysal, librosa, lightgbm, mapclassify, matplotlib, matplotlib-venn, mediapipe, mgwr, missingno, mizani, ml_dtypes, mlx

In [59]:
import subprocess
import sys

print("Installing NumPy 2.1.3 and jedi...")
subprocess.run([sys.executable, "-m", "pip", "install", "numpy==2.1.3", "jedi", "--force-reinstall"])
print("NumPy version has been adjusted.")

Installing NumPy 2.1.3 and jedi...
NumPy version has been adjusted.


In [51]:
import subprocess
import sys

print("\n=== Checking for dependency conflicts after fixes ===")
result_check = subprocess.run(
    [sys.executable, "-m", "pip", "check"],
    capture_output=True,
    text=True
)
print(result_check.stdout)
print("RETURN CODE:", result_check.returncode)
print("ERROR:\n", result_check.stderr)


=== Checking for dependency conflicts after fixes ===
No broken requirements found.

RETURN CODE: 0
ERROR:
 


In [53]:
import subprocess
import sys

# Verify current NumPy version using pip show and print its output
result = subprocess.run([sys.executable, "-m", "pip", "show", "numpy"], capture_output=True, text=True)
print(result.stdout)

Name: numpy
Version: 2.2.0
Summary: Fundamental package for array computing in Python
Home-page: https://numpy.org
Author: Travis E. Oliphant et al.
Author-email: 
License: Copyright (c) 2005-2024, NumPy Developers.
 All rights reserved.

 Redistribution and use in source and binary forms, with or without
 modification, are permitted provided that the following conditions are
 met:

     * Redistributions of source code must retain the above copyright
        notice, this list of conditions and the following disclaimer.

     * Redistributions in binary form must reproduce the above
        copyright notice, this list of conditions and the following
        disclaimer in the documentation and/or other materials provided
        with the distribution.

     * Neither the name of the NumPy Developers nor the names of any
        contributors may be used to endorse or promote products derived
        from this software without specific prior written permission.

 THIS SOFTWARE IS PROVIDED

In [60]:
import subprocess
import sys
import time

print("Stopping previous backend instances...")
subprocess.run([sys.executable, "-m", "pip", "install", "numpy==2.1.3"], check=False)
subprocess.run(["pkill", "-f", "uvicorn"], check=False)
subprocess.run(["pkill", "-f", "main.py"], check=False)
time.sleep(2)

print("Starting the clipping studio backend server...")
# Start main.py backend in the background
backend_process = subprocess.Popen([
    sys.executable, "/content/opensource-clipping/main.py"
])
print("Backend server process initiated. PID:", backend_process.pid)

Stopping previous backend instances...
Starting the clipping studio backend server...
Backend server process initiated. PID: 46792


In [61]:
import sys
sys.path.insert(0, "/content/opensource-clipping")

try:
    import clipping.studio
    print("✅ Success: clipping.studio imported cleanly with no errors!")
except Exception as e:
    import traceback
    print("❌ Error still persists:")
    traceback.print_exc()

❌ Error still persists:


Traceback (most recent call last):
  File "/tmp/ipykernel_4890/3231709153.py", line 5, in <cell line: 0>
    import clipping.studio
  File "/content/opensource-clipping/clipping/studio.py", line 38, in <module>
    _core = _load_studio_internal_module("core.py", "clipping_studio_core")
  File "/content/opensource-clipping/clipping/studio.py", line 32, in _load_studio_internal_module
    spec.loader.exec_module(module)
    ~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^
  File "/content/opensource-clipping/clipping/studio/core.py", line 25, in <module>
    import mediapipe as mp
  File "/usr/local/lib/python3.13/dist-packages/mediapipe/__init__.py", line 15, in <module>
    import mediapipe.tasks.python as tasks
  File "/usr/local/lib/python3.13/dist-packages/mediapipe/tasks/python/__init__.py", line 17, in <module>
    from . import audio
  File "/usr/local/lib/python3.13/dist-packages/mediapipe/tasks/python/audio/__init__.py", line 23, in <module>
    RunningMode = core.audio_task_running_mode.AudioT

In [63]:
import sys

patch_code = """
# Hotfix for MediaPipe on Python 3.13
import sys
try:
    import mediapipe.tasks.python.audio.core.audio_task_running_mode
    import mediapipe.tasks.python.audio.audio_classifier
    sys.modules['mediapipe.tasks.python.audio.core'] = sys.modules.get('mediapipe.tasks.python.audio.core', sys.modules['mediapipe.tasks.python.audio.core.audio_task_running_mode'])
except Exception:
    pass
"""

main_path = "/content/opensource-clipping/main.py"
with open(main_path, "r") as f:
    original_main = f.read()

if "mediapipe.tasks.python.audio" not in original_main:
    with open(main_path, "w") as f:
        f.write(patch_code + "\n" + original_main)
    print("✅ Successfully patched main.py for MediaPipe Python 3.13 compatibility!")
else:
    print("Patch already present in main.py.")

✅ Successfully patched main.py for MediaPipe Python 3.13 compatibility!


In [64]:
import sys
import importlib

# Dynamic hotfix for mediapipe's python 3.13 issue with absolute/relative imports in task package
try:
    # Pre-import the missing submodules so they are populated in sys.modules
    import mediapipe.tasks.python.audio.audio_classifier as _ac
    import mediapipe.tasks.python.audio.core.audio_task_running_mode as _rm
    sys.modules['mediapipe.tasks.python.audio.audio_classifier'] = _ac
    sys.modules['mediapipe.tasks.python.audio.core.audio_task_running_mode'] = _rm
except Exception:
    pass

# We will write a runtime wrapper/patcher into the main entry point to make sure it loads
patch_code = """
import sys
try:
    # Pre-load problematic mediapipe internals directly into sys.modules to prevent circular NameErrors on Python 3.13
    import mediapipe.tasks.python.audio.audio_classifier
    import mediapipe.tasks.python.audio.core.audio_task_running_mode
except Exception:
    pass
"""

# Prepend this patch code to main.py so it runs at the absolute beginning of the server startup
main_path = "/content/opensource-clipping/main.py"
with open(main_path, "r") as f:
    original_main = f.read()

if "mediapipe.tasks.python.audio" not in original_main:
    with open(main_path, "w") as f:
        f.write(patch_code + "\n" + original_main)
    print("Backend server main.py successfully patched for Python 3.13 compatibility!")
else:
    print("Patch already present in main.py.")

Patch already present in main.py.


In [62]:
import sys
sys.path.insert(0, "/content/opensource-clipping")

try:
    import clipping.studio
    print("✅ Success: clipping.studio imported cleanly with no errors!")
except Exception as e:
    import traceback
    print("❌ Error still persists:")
    traceback.print_exc()

❌ Error still persists:


Traceback (most recent call last):
  File "/tmp/ipykernel_4890/3231709153.py", line 5, in <cell line: 0>
    import clipping.studio
  File "/content/opensource-clipping/clipping/studio.py", line 38, in <module>
    _core = _load_studio_internal_module("core.py", "clipping_studio_core")
  File "/content/opensource-clipping/clipping/studio.py", line 32, in _load_studio_internal_module
    spec.loader.exec_module(module)
    ~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^
  File "/content/opensource-clipping/clipping/studio/core.py", line 25, in <module>
    import mediapipe as mp
  File "/usr/local/lib/python3.13/dist-packages/mediapipe/__init__.py", line 15, in <module>
    import mediapipe.tasks.python as tasks
  File "/usr/local/lib/python3.13/dist-packages/mediapipe/tasks/python/__init__.py", line 17, in <module>
    from . import audio
  File "/usr/local/lib/python3.13/dist-packages/mediapipe/tasks/python/audio/__init__.py", line 20, in <module>
    AudioClassifier = audio_classifier.AudioClassifie

In [65]:
import sys
import importlib

# Force load the missing submodules directly into Python's sys.modules registry
try:
    import mediapipe.tasks.python.audio.audio_classifier as ac
    sys.modules['mediapipe.tasks.python.audio.audio_classifier'] = ac
    import mediapipe.tasks.python.audio.core.audio_task_running_mode as rm
    sys.modules['mediapipe.tasks.python.audio.core.audio_task_running_mode'] = rm
    sys.modules['mediapipe.tasks.python.audio.core'] = rm
except Exception as e:
    print("Pre-import mapping failed:", e)

sys.path.insert(0, "/content/opensource-clipping")
try:
    import clipping.studio
    print("🎉 SUCCESS: clipping.studio imported perfectly with zero errors!")
except Exception as e:
    import traceback
    print("❌ Traceback on import:")
    traceback.print_exc()

Pre-import mapping failed: name 'audio_classifier' is not defined
❌ Traceback on import:


Traceback (most recent call last):
  File "/tmp/ipykernel_4890/777159793.py", line 16, in <cell line: 0>
    import clipping.studio
  File "/content/opensource-clipping/clipping/studio.py", line 38, in <module>
    _core = _load_studio_internal_module("core.py", "clipping_studio_core")
  File "/content/opensource-clipping/clipping/studio.py", line 32, in _load_studio_internal_module
    spec.loader.exec_module(module)
    ~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^
  File "/content/opensource-clipping/clipping/studio/core.py", line 25, in <module>
    import mediapipe as mp
  File "/usr/local/lib/python3.13/dist-packages/mediapipe/__init__.py", line 15, in <module>
    import mediapipe.tasks.python as tasks
  File "/usr/local/lib/python3.13/dist-packages/mediapipe/tasks/python/__init__.py", line 17, in <module>
    from . import audio
  File "/usr/local/lib/python3.13/dist-packages/mediapipe/tasks/python/audio/__init__.py", line 20, in <module>
    AudioClassifier = audio_classifier.AudioClassifie

In [66]:
import sys
import importlib

# Force load the missing submodules directly into Python's sys.modules registry
try:
    import mediapipe.tasks.python.audio.audio_classifier as ac
    sys.modules['mediapipe.tasks.python.audio.audio_classifier'] = ac
    import mediapipe.tasks.python.audio.core.audio_task_running_mode as rm
    sys.modules['mediapipe.tasks.python.audio.core.audio_task_running_mode'] = rm
    sys.modules['mediapipe.tasks.python.audio.core'] = rm
except Exception as e:
    print("Pre-import mapping failed:", e)

sys.path.insert(0, "/content/opensource-clipping")
try:
    import clipping.studio
    print("🎉 SUCCESS: clipping.studio imported perfectly with zero errors!")
except Exception as e:
    import traceback
    print("❌ Traceback on import:")
    traceback.print_exc()

Pre-import mapping failed: name 'audio_classifier' is not defined
❌ Traceback on import:


Traceback (most recent call last):
  File "/tmp/ipykernel_4890/777159793.py", line 16, in <cell line: 0>
    import clipping.studio
  File "/content/opensource-clipping/clipping/studio.py", line 38, in <module>
    _core = _load_studio_internal_module("core.py", "clipping_studio_core")
  File "/content/opensource-clipping/clipping/studio.py", line 32, in _load_studio_internal_module
    spec.loader.exec_module(module)
    ~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^
  File "/content/opensource-clipping/clipping/studio/core.py", line 25, in <module>
    import mediapipe as mp
  File "/usr/local/lib/python3.13/dist-packages/mediapipe/__init__.py", line 15, in <module>
    import mediapipe.tasks.python as tasks
  File "/usr/local/lib/python3.13/dist-packages/mediapipe/tasks/python/__init__.py", line 17, in <module>
    from . import audio
  File "/usr/local/lib/python3.13/dist-packages/mediapipe/tasks/python/audio/__init__.py", line 20, in <module>
    AudioClassifier = audio_classifier.AudioClassifie

In [67]:
import os
import sys

mediapipe_audio_init_path = "/usr/local/lib/python3.13/dist-packages/mediapipe/tasks/python/audio/__init__.py"

if os.path.exists(mediapipe_audio_init_path):
    with open(mediapipe_audio_init_path, "r") as f:
        content = f.read()

    # Replace the problematic circular reference with safe absolute imports
    old_code = "from mediapipe.tasks.python.audio import audio_classifier\nfrom mediapipe.tasks.python.audio import audio_embedder\n\nAudioClassifier = audio_classifier.AudioClassifier\nAudioClassifierOptions = audio_classifier.AudioClassifierOptions\nAudioEmbedder = audio_embedder.AudioEmbedder\nAudioEmbedderOptions = audio_embedder.AudioEmbedderOptions\nRunningMode = core.audio_task_running_mode.AudioTaskRunningMode"

    new_code = "try:\n    from mediapipe.tasks.python.audio.audio_classifier import AudioClassifier, AudioClassifierOptions\nexcept Exception:\n    AudioClassifier, AudioClassifierOptions = None, None\n\ntry:\n    from mediapipe.tasks.python.audio.audio_embedder import AudioEmbedder, AudioEmbedderOptions\nexcept Exception:\n    AudioEmbedder, AudioEmbedderOptions = None, None\n\ntry:\n    from mediapipe.tasks.python.audio.core.audio_task_running_mode import AudioTaskRunningMode as RunningMode\nexcept Exception:\n    RunningMode = None"

    if "audio_classifier.AudioClassifier" in content:
        patched_content = content.replace(old_code, new_code)
        with open(mediapipe_audio_init_path, "w") as f:
            f.write(patched_content)
        print("✅ Successfully patched MediaPipe's audio init for Python 3.13!")
    else:
        print("ℹ️ MediaPipe audio init patch is already applied or format differs.")
else:
    print("❌ MediaPipe path not found.")

try:
    import importlib
    import mediapipe
    importlib.reload(mediapipe)
    sys.path.insert(0, "/content/opensource-clipping")
    import clipping.studio
    print("🎉 SUCCESS! clipping.studio imported perfectly with zero errors!")
except Exception as e:
    import traceback
    print("❌ Traceback on import:")
    traceback.print_exc()

✅ Successfully patched MediaPipe's audio init for Python 3.13!
❌ Traceback on import:


Traceback (most recent call last):
  File "/tmp/ipykernel_4890/2831794273.py", line 27, in <cell line: 0>
    import mediapipe
  File "/usr/local/lib/python3.13/dist-packages/mediapipe/__init__.py", line 15, in <module>
    import mediapipe.tasks.python as tasks
  File "/usr/local/lib/python3.13/dist-packages/mediapipe/tasks/python/__init__.py", line 17, in <module>
    from . import audio
  File "/usr/local/lib/python3.13/dist-packages/mediapipe/tasks/python/audio/__init__.py", line 20, in <module>
    AudioClassifier = audio_classifier.AudioClassifier
                      ^^^^^^^^^^^^^^^^
NameError: name 'audio_classifier' is not defined


In [69]:
import sys
import os

mediapipe_audio_init_path = "/usr/local/lib/python3.13/dist-packages/mediapipe/tasks/python/audio/__init__.py"

if os.path.exists(mediapipe_audio_init_path):
    # Overwrite with clean, fully qualified absolute imports compatible with Python 3.13
    patched_content = """# Copyright 2022 The MediaPipe Authors.
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#      http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.
\"\"\"MediaPipe Tasks Audio API.\"\"\"

try:
    from mediapipe.tasks.python.audio.audio_classifier import AudioClassifier, AudioClassifierOptions
except Exception:
    AudioClassifier, AudioClassifierOptions = None, None

try:
    from mediapipe.tasks.python.audio.audio_embedder import AudioEmbedder, AudioEmbedderOptions
except Exception:
    AudioEmbedder, AudioEmbedderOptions = None, None

try:
    from mediapipe.tasks.python.audio.core.audio_task_running_mode import AudioTaskRunningMode as RunningMode
except Exception:
    RunningMode = None

# Remove duplicate refs to prevent NameErrors
__all__ = [
    'AudioClassifier',
    'AudioClassifierOptions',
    'AudioEmbedder',
    'AudioEmbedderOptions',
    'RunningMode',
]
"""
    with open(mediapipe_audio_init_path, "w") as f:
        f.write(patched_content)
    print("✅ Successfully forced direct absolute imports on MediaPipe audio init!")
else:
    print("❌ MediaPipe audio init path not found.")

# Clear cached modules to ensure the changes are reloaded from disk
for mod in list(sys.modules.keys()):
    if mod.startswith("mediapipe") or mod.startswith("clipping"):
        sys.modules.pop(mod, None)

# Restart Python session elements and verify imports
try:
    sys.path.insert(0, "/content/opensource-clipping")
    import mediapipe
    import clipping.studio
    print("🎉 HUGE SUCCESS! clipping.studio imported perfectly with zero errors!")
except Exception as e:
    import traceback
    print("❌ Traceback on import:")
    traceback.print_exc()

✅ Successfully forced direct absolute imports on MediaPipe audio init!
🎉 HUGE SUCCESS! clipping.studio imported perfectly with zero errors!


In [70]:
import subprocess
import sys
import time

print("Stopping any stale backend instances...")
subprocess.run(["pkill", "-f", "uvicorn"], check=False)
subprocess.run(["pkill", "-f", "main.py"], check=False)
time.sleep(2)

print("Starting the clipping studio backend server in the background...")
backend_process = subprocess.Popen([
    sys.executable, "/content/opensource-clipping/main.py"
])

time.sleep(5)
print(f"Backend process successfully started. PID: {backend_process.pid} | Running: {backend_process.poll() is None}")

Stopping any stale backend instances...
Starting the clipping studio backend server in the background...
Backend process successfully started. PID: 47574 | Running: True


In [55]:
import subprocess
import time
import re

# Ensure any previous cloudflared processes are killed first
subprocess.run(["pkill", "-f", "cloudflared"], check=False)
time.sleep(1)

# Start Cloudflare Tunnel and log its output
log_file_path = "/tmp/cloudflared_studio.log"
with open(log_file_path, "w") as log_file:
    tunnel_process = subprocess.Popen([
        "cloudflared", "tunnel",
        "--url", "http://127.0.0.1:8000"
    ], stdout=log_file, stderr=subprocess.STDOUT, start_new_session=True)

print("Starting Cloudflare Tunnel...")
time.sleep(8)

# Read the logs and find the generated trycloudflare URL
with open(log_file_path, "r") as f:
    log_content = f.read()

match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", log_content)
if match:
    print("\n" + "="*60)
    print("🎉 YOUR PUBLIC STUDIO BACKEND URL:")
    print(match.group(0))
    print("="*60)
    print("\n1. Copy the URL above.")
    print("2. Paste it in the 'API URL' box on the OpenSource Clipping Studio site.")
else:
    print("Could not extract tunnel URL yet. Log output:")
    print(log_content)

Starting Cloudflare Tunnel...

🎉 YOUR PUBLIC STUDIO BACKEND URL:
https://route-throw-shark-licensing.trycloudflare.com

1. Copy the URL above.
2. Paste it in the 'API URL' box on the OpenSource Clipping Studio site.


In [56]:
import os

root = "/content/opensource-clipping"
print("Searching for references to 'core' in python files...")

for folder, dirs, files in os.walk(root):
    dirs[:] = [d for d in dirs if d not in [".git", "__pycache__", ".venv"]]
    for file in files:
        if file.endswith(".py"):
            path = os.path.join(folder, file)
            try:
                with open(path, "r", encoding="utf-8", errors="ignore") as f:
                    for line_no, line in enumerate(f, 1):
                        # Search for occurrences of 'core.' or raw 'core' as a name
                        if "core" in line:
                            # Print lines that reference core to diagnose the missing import
                            print(f"{path}:{line_no}: {line.strip()}")
            except Exception as e:
                pass

Searching for references to 'core' in python files...
/content/opensource-clipping/web/api/worker.py:344: viral_score=entry.get("viral_score"),
/content/opensource-clipping/web/api/models.py:139: viral_score: Optional[int] = None
/content/opensource-clipping/clipping/voiceover.py:74: """Async core for TTS generation."""
/content/opensource-clipping/clipping/engine.py:641: - Urutkan klip berdasarkan viral_score tertinggi (paling berpotensi viral) ke terendah. Peringkat ("rank") hanya sebagai nomor urut (1, 2, 3...).
/content/opensource-clipping/clipping/engine.py:648: - Evaluasi kekuatan viral (viral-bility) dan berikan "viral_score" (1-100) yang merepresentasikan seberapa viral suatu klip.
/content/opensource-clipping/clipping/engine.py:682: Nilai viral_score 1-100 berdasarkan komponen berikut. Ini hanya untuk penilaian internal, JANGAN menambahkan field baru ke JSON.
/content/opensource-clipping/clipping/engine.py:695: - Jangan pilih klip dengan viral_score di bawah 70 kecuali jumlah 

In [71]:
import subprocess
import urllib.request
import json

print("=== 1. Checking background python processes ===")
ps_result = subprocess.run(["ps", "aux"], capture_output=True, text=True)
for line in ps_result.stdout.splitlines():
    if "main.py" in line or "uvicorn" in line or "cloudflared" in line:
        print(line)

print("\n=== 2. Testing direct connection to local backend port 8000 ===")
try:
    with urllib.request.urlopen("http://127.0.0.1:8000/", timeout=3) as response:
        print(f"Local test status code: {response.status}")
        print("Local response:", response.read().decode())
except Exception as e:
    print(f"Local test failed to connect: {e}")

=== 1. Checking background python processes ===
root       24917  0.0  0.0      0     0 ?        Zs   07:06   0:04 [cloudflared] <defunct>
root       44035  0.1  0.2 1295256 38884 ?       Ssl  08:20   0:01 cloudflared tunnel --url http://127.0.0.1:8000

=== 2. Testing direct connection to local backend port 8000 ===
Local test failed to connect: <urlopen error [Errno 111] Connection refused>


In [72]:
import subprocess
import sys
import time

print("Killing existing instances...")
subprocess.run(["pkill", "-f", "uvicorn"], check=False)
subprocess.run(["pkill", "-f", "main.py"], check=False)
time.sleep(2)

print("Launching backend server in background, logging to /tmp/backend.log...")
log_file = open("/tmp/backend.log", "w")
backend = subprocess.Popen([
    sys.executable, "/content/opensource-clipping/main.py"
], stdout=log_file, stderr=subprocess.STDOUT)

time.sleep(5)
print("Backend started (PID):", backend.pid)

with open("/tmp/backend.log", "r") as f:
    print("\n--- Backend Logs ---")
    print(f.read())

Killing existing instances...
Launching backend server in background, logging to /tmp/backend.log...
Backend started (PID): 48680

--- Backend Logs ---
2026-09-28 08:38:11.355008: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
usage: main.py [-h] [--url URL [URL ...]] [--yt-cookies YT_COOKIES]
               [--source {youtube,tiktok,instagram,gdrive}] [--tiktok]
               [--clips CLIPS] [--ratio {9:16,16:9,1:1,3:4,4:5}]
               [--source-height SOURCE_HEIGHT] [--render-height RENDER_HEIGHT]
               [--words-per-sub WORDS_PER_SUB] [--hook-duration HOOK_DURATION]
               [--hook-source HOOK_SOURCE]
               [--hook-source-start HOOK_SOURCE_START] [--no-broll]
               [--no-hook] [--cleanup-source] [--no-bgm

In [78]:
import subprocess
import sys
import time
import urllib.request

print("Stopping any broken background processes...")
subprocess.run(["pkill", "-f", "uvicorn"], check=False)
subprocess.run(["pkill", "-f", "main.py"], check=False)
time.sleep(2)

print("Launching the FastAPI Web Server via Uvicorn in the background...")
log_file = open("/tmp/backend.log", "w")
backend = subprocess.Popen([
    sys.executable, "-m", "uvicorn",
    "web.api.app:app",
    "--host", "0.0.0.0",
    "--port", "8000",
    "--log-level", "info"
], stdout=log_file, stderr=subprocess.STDOUT)

time.sleep(5)
print(f"Web server process started with PID: {backend.pid}")

print("\n=== Checking Local Health Check (Port 8000) ===")
try:
    with urllib.request.urlopen("http://127.0.0.1:8000/", timeout=5) as response:
        print(f"Success! HTTP status code: {response.status}")
except Exception as e:
    print(f"Local connection test failed: {e}")
    with open("/tmp/backend.log", "r") as f:
        print("\n--- Backend Startup Logs ---")
        print(f.read())

Stopping any broken background processes...
Launching the FastAPI Web Server via Uvicorn in the background...
Web server process started with PID: 69163

=== Checking Local Health Check (Port 8000) ===
Success! HTTP status code: 200


### ⚡ GPU Acceleration & Performance Optimization
Running deep learning models (Whisper, Face Detection) and rendering videos can be accelerated significantly using Colab's GPU runtime.

In [74]:
import torch
import subprocess

# 1. Verify GPU Presence
gpu_available = torch.cuda.is_available()
if gpu_available:
    device_name = torch.cuda.get_device_name(0)
    print(f"✅ Active GPU Detected: {device_name}")
    print("Backend will automatically utilize CUDA for Whisper and model-based operations.")
else:
    print("⚠️ GPU is not active. Go to 'Runtime' -> 'Change runtime type' -> Select 'T4 GPU' to speed up generation!")

# 2. Check for hardware-accelerated ffmpeg encoders
print("\nChecking for hardware-accelerated video encoders...")
result = subprocess.run(["ffmpeg", "-encoders"], capture_output=True, text=True)
if "h264_nvenc" in result.stdout:
    print("✅ NVENC (NVIDIA Hardware Encoding) is supported by FFmpeg!")
else:
    print("ℹ️ Standard CPU encoders (libx264) will be used for rendering.")

✅ Active GPU Detected: Tesla T4
Backend will automatically utilize CUDA for Whisper and model-based operations.

Checking for hardware-accelerated video encoders...
✅ NVENC (NVIDIA Hardware Encoding) is supported by FFmpeg!


In [57]:
with open("/content/opensource-clipping/clipping/studio.py", "r") as f:
    content = f.read()

# Let's inspect the dynamic import helper and surrounding code
lines = content.splitlines()
for idx, line in enumerate(lines[:80], 1):
    print(f"{idx:02d}: {line}")

01: """
02: clipping.studio — Video Rendering Engine
03: 
04: Thin compatibility entry point for the Studio pipeline.
05: Public API is re-exported from internal modules under clipping/studio/.
06: """
07: 
08: import importlib.util
09: import os
10: 
11: FIREFOX_UA = (
12:     "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:148.0) Gecko/20100101 Firefox/148.0"
13: )
14: 
15: 
16: def _load_studio_internal_module(file_name: str, module_alias: str):
17:     """
18:     Load an internal `clipping/studio/*.py` module by file path.
19: 
20:     Args:
21:         file_name: Python filename inside `clipping/studio`.
22:         module_alias: Unique import alias used by importlib.
23: 
24:     Returns:
25:         Loaded Python module object.
26:     """
27:     module_path = os.path.join(os.path.dirname(__file__), "studio", file_name)
28:     spec = importlib.util.spec_from_file_location(module_alias, module_path)
29:     if spec is None or spec.loader is None:
30:         raise ImportError(f"

In [58]:
import sys
sys.path.insert(0, "/content/opensource-clipping")

try:
    import clipping.studio
    print("clipping.studio imported successfully!")
except Exception as e:
    import traceback
    print("Error importing clipping.studio:")
    traceback.print_exc()

Error importing clipping.studio:


Traceback (most recent call last):
  File "/tmp/ipykernel_4890/393661390.py", line 5, in <cell line: 0>
    import clipping.studio
  File "/content/opensource-clipping/clipping/studio.py", line 38, in <module>
    _core = _load_studio_internal_module("core.py", "clipping_studio_core")
  File "/content/opensource-clipping/clipping/studio.py", line 32, in _load_studio_internal_module
    spec.loader.exec_module(module)
    ~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^
  File "/content/opensource-clipping/clipping/studio/core.py", line 25, in <module>
    import mediapipe as mp
  File "/usr/local/lib/python3.13/dist-packages/mediapipe/__init__.py", line 15, in <module>
    import mediapipe.tasks.python as tasks
  File "/usr/local/lib/python3.13/dist-packages/mediapipe/tasks/python/__init__.py", line 17, in <module>
    from . import audio
  File "/usr/local/lib/python3.13/dist-packages/mediapipe/tasks/python/audio/__init__.py", line 18, in <module>
    import mediapipe.tasks.python.audio.audio_classifi